# Prepare MAGFiLO archive for analysis

The connected Google Drive ZIP is ~704 MB, while the connector can retrieve files of at most 256 MiB.

Run **Runtime → Run all**, then authorize Google Drive with the same account that holds **بيانات سولر**. This will create four smaller `partNN` files and a checksum manifest alongside the original ZIP. The original ZIP stays intact. **No files are made public.**

When finished, return to ChatGPT and say that splitting is complete. This step is for data access; model training still requires Kaggle GPU or another GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import math
import os

root = Path('/content/drive/MyDrive')
archive = root / 'بيانات سولر' / 'filament-segmentation-2026.zip'
if not archive.is_file():
    raise FileNotFoundError(f'Cannot find {archive}; please check the active Google account.')

chunk_size = 180 * 1024 * 1024  # Safe under the connector's 256 MiB limit
buffer_size = 8 * 1024 * 1024
source_size = archive.stat().st_size
n_parts = math.ceil(source_size / chunk_size)
if n_parts < 1:
    raise ValueError('The archive is empty.')

parts = []
whole_digest = sha256()
with archive.open('rb') as source:
    for index in range(1, n_parts + 1):
        part = archive.with_name(f'{archive.name}.part{index:02d}')
        temporary = archive.with_name(f'{archive.name}.part{index:02d}.tmp')
        expected_bytes = min(chunk_size, source_size - (index - 1) * chunk_size)
        remaining = expected_bytes
        part_digest = sha256()
        try:
            with temporary.open('wb') as destination:
                while remaining:
                    data = source.read(min(buffer_size, remaining))
                    if not data:
                        raise IOError('Source ZIP ended unexpectedly')
                    destination.write(data)
                    part_digest.update(data)
                    whole_digest.update(data)
                    remaining -= len(data)
            os.replace(temporary, part)
        finally:
            if temporary.exists():
                temporary.unlink()
        parts.append({'name': part.name, 'bytes': expected_bytes, 'sha256': part_digest.hexdigest()})
        print(f'{index}/{n_parts} completed: {part.name} ({expected_bytes / 1048576:.1f} MiB)', flush=True)

manifest = {
    'original_file': archive.name,
    'original_size': source_size,
    'original_sha256': whole_digest.hexdigest(),
    'parts': parts,
    'chunk_size': chunk_size,
}
manifest_path = archive.with_name('filament-segmentation-2026.parts.json')
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'\nFinished. {len(parts)} parts and manifest saved in: {archive.parent}')
print('Now return to ChatGPT and say: Finished splitting the MAGFiLO dataset.')